# Axis 3 — Prefill Granularity & Expert Amortisation (Mac / MLX)

**Question.** Which prefill chunk size C gets the most reuse out of each expert load without blowing the memory budget?

**Hypothesis (from the team page).** As C grows from 1 to 128, TTFT drops non-linearly because tokens in the same chunk share experts (the chance that an expert loaded for one token is reused by another approaches 1). Beyond C = 128 the set of distinct experts a chunk needs exceeds the cache, forcing scratch-memory growth or evictions inside the chunk.

## Why chunking matters with an expert cache
A prefill pass over C tokens asks each layer for the *union* of the experts its C tokens route to. One load then serves many tokens (amortisation). But the pass walks all 16 layers before the next chunk starts, so the experts of one chunk across all layers must fit in K, or layer 0's experts are evicted by layer 15 before the next chunk reuses them.

## What is measured, for each (C, K)
| Metric | Meaning |
|---|---|
| TTFT | prefill of the whole prompt + first token, cold cache |
| prefill I/O | experts loaded, GB read, time in disk reads |
| reuse ratio | token→expert assignments ÷ distinct experts loaded, per layer call |
| distinct / layer call | experts one layer needs for one chunk (max 64) |
| chunk footprint | Σ over 16 layers of distinct experts per chunk, vs K |
| grouped calls | layer calls needing more experts than K: processed in groups of ≤ K (intra-chunk eviction) |
| scratch memory | MLX peak minus bank minus weights (activations, attention, KV) |

Everything is a real run on the disk-backed cache; no simulation.

**Runtime.** Full grid ≈ 40–60 min (C = 1 is slow by design). `MOE_SMOKE=1` takes a few minutes.

In [ ]:
import os, sys, time, json, importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlx.core as mx

# Find the repo and import the shared module (experiments/moe_cache.py).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'experiments' / 'moe_cache.py').exists())
sys.path.insert(0, str(ROOT / 'experiments'))
import moe_cache as mc
importlib.reload(mc)

# MOE_SMOKE=1 runs a tiny configuration end to end (used to test the notebook).
SMOKE = os.environ.get('MOE_SMOKE') == '1'
AXIS_DIR = ROOT / 'experiments' / 'axis-3-prefill-chunking'
RESULTS = (ROOT / 'experiments' / '.cache' / 'smoke' / 'axis-3-prefill-chunking') if SMOKE else (AXIS_DIR / 'results')
RESULTS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.figsize': (11, 5), 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})
print('SMOKE run' if SMOKE else 'Full run', '| results ->', RESULTS.relative_to(ROOT))
print(mc.system_info())

## Configuration

In [ ]:
CHUNKS = [1, 4, 16, 32, 64, 128, 256, 512]     # plus 'full' = the whole prompt in one pass
K_VALUES = [128, 256, 512]
PROMPT_TOKENS = 512            # every prompt is trimmed to this many article tokens
N_PROMPTS = 2                  # long passages written for this axis (max 2)
INCLUDE_FULL = True

if SMOKE:
    CHUNKS, K_VALUES, PROMPT_TOKENS, N_PROMPTS = [8, 64], [128, 512], 128, 1

print('chunks:', CHUNKS + (['full'] if INCLUDE_FULL else []), '| K:', K_VALUES,
      '| prompt tokens:', PROMPT_TOKENS)

## Model and correctness gate

In [ ]:
model, tokenizer, index = mc.load_model()
print(index.describe())
print(f'Non-expert weights resident: {mc.non_expert_bytes(model) / 1e9:.2f} GB '
      f'| MLX active memory: {mx.get_active_memory() / 1e9:.2f} GB')
print(f'Largest K that fits safely in this machine\'s RAM: {mc.max_safe_capacity(index)}')

# Correctness gate: one layer through the K-slot bank vs the stock mlx-lm layer
# (all 64 experts of that layer stacked). Must be bit-identical.
diff = mc.check_layer_exactness(model, index, capacity=64)
assert diff == 0.0, diff
print(f'Layer exactness vs stock mlx-lm: max |diff| = {diff}')

## Long prompts (fixed length)
Two long passages written for this axis, each trimmed to exactly `PROMPT_TOKENS` tokens so every chunk size processes the same input.

In [ ]:
# Long inputs written for this axis: prefill only matters when the prompt is long.
PASSAGES = [
'''When a city grows, its water system grows with it, usually in pieces and rarely according to a single plan.
The oldest districts are often served by cast-iron mains laid more than a century ago, while new suburbs receive
ductile iron or plastic pipes sized for populations that planners hoped would arrive. Water usually begins its
journey in a reservoir or a river intake, where screens remove leaves, fish and debris. It then passes through a
treatment plant. There, chemicals called coagulants are added so that tiny particles of clay and organic matter
clump together into larger flakes. These flakes settle to the bottom of wide, slow tanks, and the clearer water on
top flows onward through beds of sand and gravel that trap whatever remains. Finally the water is disinfected,
most often with chlorine, sometimes with ultraviolet light or ozone, so that bacteria and viruses cannot survive
the trip to a kitchen tap. A small amount of chlorine is deliberately left in the water so that it keeps protecting
it inside the pipes. From the plant, pumps push the water into a network of mains and into storage towers or
hilltop tanks. These tanks are not decoration: they let the utility run its pumps steadily through the night,
when electricity is cheap and demand is low, and then release water by gravity during the morning rush, when
everyone showers at once. They also keep pressure in the system if a pump fails and provide a reserve for
firefighters. Pressure is a constant balancing act. Too little, and taps on upper floors run dry and dirty
groundwater can seep into cracked pipes. Too much, and old joints burst and leaks multiply. Engineers divide the
network into pressure zones, separated by valves and regulators, so that a house at the bottom of a hill does not
receive the same crushing pressure as one at the top. Leaks are the quiet enemy of every utility. In many cities
a fifth or more of the treated water never reaches a customer; it seeps away underground, unseen, until a road
collapses or a basement floods. Utilities hunt for leaks with acoustic sensors that listen for the hiss of
escaping water, with satellite images that reveal damp soil, and increasingly with data: if the flow into a
district at three in the morning, when almost everyone is asleep, is much higher than it should be, water is
probably escaping somewhere. Replacing pipes is slow and expensive, because every repair means digging up a
street, diverting traffic and interrupting service to homes and businesses. As a result, cities must decide which
pipes to replace first, weighing their age, material, history of breaks and the damage a failure would cause.
Climate adds another layer of uncertainty. Longer droughts shrink reservoirs, heavier storms overwhelm drains and
treatment plants, and warmer water encourages algae that make treatment harder. Some cities now recycle
wastewater into drinking water after advanced purification, while others capture rainwater or desalinate
seawater. Each option trades money, energy and public trust differently, and the right mix depends on local
geography, politics and how much people are willing to pay for a resource they mostly take for granted.''',
'''For most of history, sailors judged their position at sea by a mixture of skill, guesswork and luck. Close to
shore they could follow landmarks, measure the depth with a weighted line and read the color of the water. Far
from land these clues disappeared, and navigators relied on dead reckoning: if you know where you started, how
fast you are moving and in which direction, you can estimate where you are now. The method sounds simple, but
small errors accumulate. A current that pushes the ship slightly sideways, a compass that is a degree off, or a
speed that is overestimated by a single knot can place a ship miles from where its crew believes it to be after
a few days. Latitude, the distance north or south of the equator, was the easier half of the problem. By
measuring the angle of the sun at noon or of the pole star at night, a navigator with a simple instrument and a
table of figures could work out latitude fairly accurately. Longitude, the distance east or west, was far harder,
because the earth turns. The sky at a given moment looks the same along a whole line running from pole to pole,
so the stars alone cannot tell you how far east or west you are unless you also know the exact time at some
reference place. Every hour of difference between local noon and noon at home corresponds to fifteen degrees of
longitude, so a clock that drifts by only a few seconds a day becomes useless on a long voyage. Governments
offered large prizes for a practical solution, because errors in longitude cost ships, cargo and lives. Two
approaches competed. Astronomers proposed measuring the position of the moon against the background stars and
comparing it with predicted tables; the method worked but demanded difficult observations and long calculations.
Clockmakers proposed building a timekeeper that could survive the motion, temperature changes and humidity of a
ship at sea. After decades of patient work, one craftsman produced a series of marine clocks accurate enough to
keep time within a few seconds over weeks of sailing. These chronometers were at first rare and expensive, and
ships often carried several so that their readings could be compared. Over time they became standard equipment,
and lunar tables and chronometers were used side by side, each checking the other. In the twentieth century,
radio time signals let ships correct their clocks every day, and radio beacons offered new ways to fix a
position. Later, systems based on satellites made it possible for anyone with a small receiver to know their
position within a few meters, at any hour and in any weather. Yet navigators are still trained in older methods,
because electronic systems can fail, be jammed or give wrong answers without warning. A crew that understands
dead reckoning and can take a sight of the sun has a way to notice when the screen is lying. The long struggle
to measure longitude is often told as a story of genius, but it is also a story about verification: about
building tools whose errors can be measured, compared and corrected before they become dangerous.''',
]
prompts = []
for text in PASSAGES[:N_PROMPTS]:
    ids = tokenizer.encode(text)
    assert len(ids) >= PROMPT_TOKENS, f'passage has only {len(ids)} tokens'
    prompts.append(mc.chat_tokens(tokenizer, 'Summarize the following passage in three sentences.\n\n'
                                  + tokenizer.decode(ids[:PROMPT_TOKENS])))
print('passage tokens:', [len(tokenizer.encode(t)) for t in PASSAGES],
      '| prompt lengths used (incl. chat template):', [len(t) for t in prompts])

## Sweep
Fresh (cold) cache for every run, so each TTFT includes loading what the prompt needs. Pass `C = 'full'` is the whole prompt in one forward pass.

In [ ]:
SAFE_K = mc.max_safe_capacity(index)
K_VALUES = [k for k in K_VALUES if k <= SAFE_K]
weights_GB = mc.non_expert_bytes(model) / 1e9
chunk_list = CHUNKS + (['full'] if INCLUDE_FULL else [])
rows, call_rows = [], []
ref_tok = {}
for K in K_VALUES:
    for C in chunk_list:
        for pi, toks in enumerate(prompts):
            mc.free_memory(); mx.reset_peak_memory()
            cache = mc.ExpertCache(index, K, 'LRU')
            rec = mc.Recorder()
            mc.attach_cache(model, cache, [rec])
            chunk = None if C == 'full' else C
            on_step = lambda ph, j: rec.set(phase=ph, chunk=j)
            r = mc.generate(model, tokenizer, toks, max_new_tokens=1, prefill_chunk=chunk, on_step=on_step)
            s = cache.summary()
            peak = mx.get_peak_memory() / 1e9
            # Chunking should not change the result (it can, rarely, by float rounding).
            ref_tok.setdefault(pi, r.tokens[0])
            same_token = r.tokens[0] == ref_tok[pi]

            ev = pd.DataFrame(rec.events)
            ev['distinct'] = ev.eids.map(len)
            ev['reuse'] = ev.n * 8 / ev.distinct
            per_chunk = ev.groupby('chunk').distinct.sum()          # experts per chunk, all layers
            n_chunks = len(per_chunk)
            rows.append({
                'K': K, 'C': C, 'C_num': len(toks) if C == 'full' else C, 'prompt': pi,
                'prompt_tokens': len(toks), 'n_passes': n_chunks, 'ttft_s': r.ttft_s,
                'misses': s['misses'], 'hit_rate': s['hit_rate'], 'GB_loaded': s['GB_loaded'],
                'io_s': s['io_ms'] / 1000, 'route_wait_s': mc.route_wait_ms(model) / 1000,
                'reuse_mean': ev.reuse.mean(), 'distinct_per_call': ev.distinct.mean(),
                'distinct_per_call_max': ev.distinct.max(), 'chunk_footprint': per_chunk.mean(),
                'chunk_footprint_max': per_chunk.max(),
                'grouped_calls': int((ev.distinct > K).sum()),
                'peak_GB': peak, 'scratch_GB': peak - s['bank_GB'] - weights_GB,
                'same_first_token': same_token,
            })
            cache.close(); del cache
            print(f'K={K:4d} C={str(C):>5s} p{pi}: TTFT {r.ttft_s:6.2f}s | {n_chunks:4d} passes | '
                  f'loads {s["misses"]:5d} ({s["GB_loaded"]:.1f} GB) | reuse {ev.reuse.mean():5.1f}x | '
                  f'footprint {per_chunk.mean():6.0f} | grouped {rows[-1]["grouped_calls"]} | '
                  f'scratch {rows[-1]["scratch_GB"]:.2f} GB')

res = pd.DataFrame(rows)
res.to_csv(RESULTS / 'axis3_runs.csv', index=False)
print(f'\nFirst token identical to the first run in {res.same_first_token.mean():.0%} of runs '
      '(differences would come from float rounding in chunked attention).')

## Results

In [ ]:
g = res.groupby(['K', 'C_num']).agg(ttft_s=('ttft_s', 'mean'), GB_loaded=('GB_loaded', 'mean'),
                                     io_s=('io_s', 'mean'), reuse=('reuse_mean', 'mean'),
                                     distinct_per_call=('distinct_per_call', 'mean'),
                                     footprint=('chunk_footprint', 'mean'), grouped=('grouped_calls', 'mean'),
                                     scratch_GB=('scratch_GB', 'mean')).reset_index()
best = g.loc[g.groupby('K').ttft_s.idxmin(), ['K', 'C_num', 'ttft_s']]
print('Fastest chunk size per K:'); display(best.set_index('K'))
g.style.format({'ttft_s': '{:.2f}', 'GB_loaded': '{:.2f}', 'io_s': '{:.2f}', 'reuse': '{:.1f}',
                'distinct_per_call': '{:.1f}', 'footprint': '{:.0f}', 'grouped': '{:.0f}', 'scratch_GB': '{:.2f}'})

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for K, d in g.groupby('K'):
    axes[0, 0].plot(d.C_num, d.ttft_s, '-o', label=f'K={K}')
    axes[0, 1].plot(d.C_num, d.GB_loaded, '-o', label=f'K={K}')
    axes[1, 1].plot(d.C_num, d.scratch_GB, '-o', label=f'K={K}')
d0 = g[g.K == g.K.min()]
axes[1, 0].plot(d0.C_num, d0.reuse, '-o', color='C3', label='reuse ratio (assignments / loads)')
ax2 = axes[1, 0].twinx()
ax2.plot(d0.C_num, d0.footprint, '-s', color='C4', label='chunk footprint (experts, 16 layers)')
for K in g.K.unique():
    ax2.axhline(K, color='C4', ls=':', alpha=0.6); ax2.text(d0.C_num.max(), K, f' K={K}', fontsize=8, color='C4')
ax2.set_ylabel('experts per chunk'); axes[1, 0].set_ylabel('reuse ratio')
axes[1, 0].legend(loc='upper left', fontsize=8); ax2.legend(loc='lower right', fontsize=8)
titles = ['TTFT (cold cache)', 'Expert bytes read during prefill', 'Amortisation vs chunk footprint',
          'Scratch memory (peak - bank - weights)']
ylabels = ['s', 'GB', None, 'GB']
for ax, t, yl in zip(axes.flat, titles, ylabels):
    ax.set_xscale('log', base=2); ax.set_xlabel('chunk size C (tokens)'); ax.set_title(t)
    ax.axvline(128, color='grey', ls='--', alpha=0.5)
    if yl: ax.set_ylabel(yl)
for ax in [axes[0, 0], axes[0, 1], axes[1, 1]]:
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(RESULTS / 'axis3_chunking.png', dpi=150, bbox_inches='tight'); plt.show()

## How many distinct experts does one layer need per chunk?
If routing were uniform, a layer would need $64\,(1-(1-8/64)^C)$ distinct experts for C tokens. The measured curve shows how much faster than "uniform" real routing saturates.

In [ ]:
d0 = g[g.K == g.K.max()].sort_values('C_num')
C = d0.C_num.values
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(C, d0.distinct_per_call, '-o', label='measured')
ax.plot(C, 64 * (1 - (1 - 8 / 64) ** C), '--', label='uniform routing')
ax.axhline(64, color='grey', ls=':'); ax.set_xscale('log', base=2)
ax.set_xlabel('chunk size C'); ax.set_ylabel('distinct experts per layer call'); ax.legend()
ax.set_title('Distinct experts per layer vs chunk size')
fig.savefig(RESULTS / 'axis3_distinct.png', dpi=150, bbox_inches='tight'); plt.show()

## Hypothesis check (descriptive)

In [ ]:
for K, d in g.groupby('K'):
    d = d.sort_values('C_num')
    t = dict(zip(d.C_num, d.ttft_s))
    c_best = int(d.loc[d.ttft_s.idxmin(), 'C_num'])
    over = d[d.footprint > K].C_num.min()
    print(f'K={K}: fastest C = {c_best} ({t[c_best]:.2f}s); '
          f'TTFT at the smallest C = {t[d.C_num.min()]:.2f}s ({t[d.C_num.min()] / t[c_best]:.1f}x slower). '
          f'Chunk footprint exceeds K from C = {over if not pd.isna(over) else "never"}; '
          f'grouped (intra-chunk-evicting) calls at the largest C: {int(d.grouped.iloc[-1])}.')
print('\nThe hypothesis predicts the fastest C near 128 and footprint > K beyond it.')